# LiteLLMMiddleware — Multi-Provider Memory Extraction

This notebook shows how to wrap **any LiteLLM-supported provider** with Smritikosh's
`LiteLLMMiddleware` so memory is extracted automatically from every conversation turn —
without changing a single line of your LLM call code.

Providers covered:
- **Ollama** — local models (Llama 3, Mistral, Qwen, Phi, …)
- **vLLM** — self-hosted OpenAI-compatible inference server
- **Google Gemini** — `gemini-1.5-flash`, `gemini-1.5-pro`, `gemini-2.0-flash`
- **OpenAI** — `gpt-4o`, `gpt-4o-mini` (via LiteLLM)
- **Anthropic Claude** — `claude-haiku-4-5-20251001`, `claude-sonnet-4-6` (via LiteLLM)

All providers share the same interface because LiteLLM normalises every response to
the OpenAI schema — so the `remember()` tool injection, turn buffering, and session
ingest all work identically regardless of which backend you use.

---

## Prerequisites

```bash
pip install litellm smritikosh
```

A Smritikosh server must be running on `http://localhost:8080`.
See the [QUICKSTART](../QUICKSTART.md) for setup instructions.


## 0 — Shared setup

Run this cell once. It sets the Smritikosh connection parameters used in every
section below. Swap `user_id` and `app_id` to match your environment.

In [ ]:
import os

import litellm
from smritikosh.sdk import LiteLLMMiddleware

SMRITIKOSH_URL = os.getenv("SMRITIKOSH_URL", "http://localhost:8080")
SMRITIKOSH_API_KEY = os.getenv("SMRITIKOSH_API_KEY", "sk-smriti-your-key-here")

# User whose memories will be extracted
USER_ID = "alice"
APP_ID  = "default"

# Turn down LiteLLM's verbose logging so notebook output stays clean
litellm.set_verbose = False

print(f"Smritikosh URL : {SMRITIKOSH_URL}")
print(f"User           : {USER_ID} / {APP_ID}")

---

## 1 — Ollama (local models)

Ollama runs models entirely on your machine — no API key, no data leaves your network.

**Install Ollama:** https://ollama.com  
**Pull a model first:** `ollama pull llama3.2` (or `mistral`, `qwen2.5`, `phi3`, …)

The model string format is `ollama_chat/<model-name>`.

In [ ]:
OLLAMA_MODEL = "ollama_chat/llama3.2"   # change to any model you have pulled
OLLAMA_BASE  = "http://localhost:11434" # default Ollama API address

conversation = [
    {"role": "user", "content": "Hi! I'm an ML engineer migrating our training cluster from PyTorch to JAX."},
    {"role": "assistant", "content": "That's a significant migration. What's driving the switch?"},
    {"role": "user", "content": "Mainly XLA compilation speed and better TPU support. Also, I always prefer Python over notebooks for anything production."},
]

with LiteLLMMiddleware(
    litellm,
    smritikosh_url=SMRITIKOSH_URL,
    smritikosh_api_key=SMRITIKOSH_API_KEY,
    user_id=USER_ID,
    app_id=APP_ID,
    extract_every_n_turns=3,   # flush after every 3 user turns
    auto_inject=True,          # prepend memory context before each LLM call
) as mw:
    response = mw.completion(
        model=OLLAMA_MODEL,
        api_base=OLLAMA_BASE,
        messages=conversation,
    )
    print("[Ollama response]")
    print(response.choices[0].message.content)
    # close() / __exit__ flushes remaining turns to Smritikosh automatically

**What happened:**
1. `LiteLLMMiddleware` injected the `remember()` tool definition into the request.
2. Ollama responded normally. If it called `remember()`, the fact was saved to
   `POST /memory/fact` transparently and a follow-up call was made.
3. Because `auto_inject=True`, Smritikosh fetched Alice's existing memory context
   and prepended it to the system message before the call.
4. On `__exit__`, all buffered turns were flushed to `POST /ingest/session`.

---

## 2 — vLLM (self-hosted OpenAI-compatible server)

vLLM exposes an OpenAI-compatible REST API. Pass `api_base` pointing to your
vLLM server and use the `openai/<model-name>` prefix.

**Start a vLLM server:**
```bash
pip install vllm
python -m vllm.entrypoints.openai.api_server \
    --model mistralai/Mistral-7B-Instruct-v0.3 \
    --port 8000
```

In [ ]:
VLLM_MODEL    = "openai/mistralai/Mistral-7B-Instruct-v0.3"  # prefix must be "openai/"
VLLM_BASE_URL = "http://localhost:8000/v1"                    # your vLLM server

conversation = [
    {"role": "user", "content": "I use Neovim with the lazy.nvim plugin manager. I switched from VS Code last year and never looked back."},
]

with LiteLLMMiddleware(
    litellm,
    smritikosh_url=SMRITIKOSH_URL,
    smritikosh_api_key=SMRITIKOSH_API_KEY,
    user_id=USER_ID,
    app_id=APP_ID,
) as mw:
    response = mw.completion(
        model=VLLM_BASE_URL and VLLM_MODEL,   # guards against running without a server
        api_base=VLLM_BASE_URL,
        messages=conversation,
        api_key="not-needed",  # vLLM doesn't require an API key by default
    )
    print("[vLLM response]")
    print(response.choices[0].message.content)

**Using llama.cpp instead of vLLM?**

Start `llama-server` with `--port 8080 --chat-template chatml` and point `api_base`
at `http://localhost:8080/v1`.  The model string and all middleware parameters stay
identical — only the `api_base` changes.

---

## 3 — Google Gemini

Get an API key at https://aistudio.google.com/app/apikey  
Available models: `gemini/gemini-1.5-flash`, `gemini/gemini-1.5-pro`, `gemini/gemini-2.0-flash`

In [ ]:
import os

GEMINI_MODEL   = "gemini/gemini-1.5-flash"  # cheapest + fastest Gemini model
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY", "")

if not GEMINI_API_KEY:
    print("Set GEMINI_API_KEY to run this cell.")
else:
    conversation = [
        {"role": "user", "content": "My goal this quarter is to ship the new recommendation engine before the Q2 board review. I always prioritise async communication — I check Slack in batches, not constantly."},
    ]

    with LiteLLMMiddleware(
        litellm,
        smritikosh_url=SMRITIKOSH_URL,
        smritikosh_api_key=SMRITIKOSH_API_KEY,
        user_id=USER_ID,
        app_id=APP_ID,
        enable_remember_tool=True,  # let Gemini call remember() when it notices something important
    ) as mw:
        response = mw.completion(
            model=GEMINI_MODEL,
            api_key=GEMINI_API_KEY,
            messages=conversation,
        )
        print("[Gemini response]")
        print(response.choices[0].message.content)

---

## 4 — OpenAI via LiteLLM

You can use `SmritikoshMiddleware(openai.OpenAI())` directly (see `middleware_demo.py`),
but `LiteLLMMiddleware` works too and lets you swap providers by just changing the
model string — zero code change.

In [ ]:
import os

OPENAI_MODEL   = "gpt-4o-mini"
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")

if not OPENAI_API_KEY:
    print("Set OPENAI_API_KEY to run this cell.")
else:
    conversation = [
        {"role": "user", "content": "I started learning Rust last month. The borrow checker finally clicked for me after I read the Ownership chapter twice."},
    ]

    with LiteLLMMiddleware(
        litellm,
        smritikosh_url=SMRITIKOSH_URL,
        smritikosh_api_key=SMRITIKOSH_API_KEY,
        user_id=USER_ID,
        app_id=APP_ID,
    ) as mw:
        response = mw.completion(
            model=OPENAI_MODEL,
            api_key=OPENAI_API_KEY,
            messages=conversation,
        )
        print("[OpenAI response]")
        print(response.choices[0].message.content)

---

## 5 — Anthropic Claude via LiteLLM

LiteLLM translates the OpenAI tool-call schema to Anthropic's `tool_use` format
transparently, so the `remember()` tool works identically.

In [ ]:
import os

CLAUDE_MODEL   = "claude-haiku-4-5-20251001"  # fastest + cheapest Claude model
ANTHROPIC_KEY  = os.getenv("ANTHROPIC_API_KEY", "")

if not ANTHROPIC_KEY:
    print("Set ANTHROPIC_API_KEY to run this cell.")
else:
    conversation = [
        {"role": "user", "content": "I prefer written async updates over video calls. I think better in text. My manager keeps scheduling 30-min standups and I always wish they were Slack threads instead."},
    ]

    with LiteLLMMiddleware(
        litellm,
        smritikosh_url=SMRITIKOSH_URL,
        smritikosh_api_key=SMRITIKOSH_API_KEY,
        user_id=USER_ID,
        app_id=APP_ID,
    ) as mw:
        response = mw.completion(
            model=CLAUDE_MODEL,
            api_key=ANTHROPIC_KEY,
            messages=conversation,
        )
        print("[Claude response]")
        print(response.choices[0].message.content)

---

## 6 — Verify extracted memories

After running any of the provider cells above, query Smritikosh to confirm that
the conversation turns were extracted and stored.

In [ ]:
import httpx

headers = {"Authorization": f"Bearer {SMRITIKOSH_API_KEY}", "Content-Type": "application/json"}

# Retrieve the 5 most recent memories for alice
r = httpx.get(
    f"{SMRITIKOSH_URL}/memory/{USER_ID}",
    params={"app_id": APP_ID, "limit": 5},
    headers=headers,
    timeout=10,
)
r.raise_for_status()

events = r.json().get("events", [])
print(f"Latest {len(events)} memories for {USER_ID}:\n")
for ev in events:
    src = ev.get("source_type", "?")
    imp = ev.get("importance_score", 0)
    text = (ev.get("raw_text") or "")[:120]
    print(f"  [{imp:.2f}] ({src}) {text}")

In [ ]:
# Search memories semantically
r = httpx.post(
    f"{SMRITIKOSH_URL}/memory/search",
    json={"user_id": USER_ID, "app_id": APP_ID, "query": "programming language preference", "top_k": 5},
    headers=headers,
    timeout=15,
)
r.raise_for_status()

results = r.json().get("results", [])
print(f'Search: "programming language preference" — {len(results)} results\n')
for res in results:
    score = res.get("score", 0)
    text  = (res.get("raw_text") or "")[:120]
    print(f"  [{score:.3f}] {text}")

In [ ]:
# Build context for the next LLM call
r = httpx.post(
    f"{SMRITIKOSH_URL}/context",
    json={"user_id": USER_ID, "app_id": APP_ID, "query": "What tools and languages does Alice use?"},
    headers=headers,
    timeout=15,
)
r.raise_for_status()

ctx = r.json()
print("Context block (inject into your LLM system prompt):\n")
print(ctx.get("context_text", "(empty)"))

procedures = ctx.get("procedures", [])
if procedures:
    print("\nMatched procedural rules:")
    for p in procedures:
        print(f"  [pri {p['priority']}] {p['trigger']} → {p['instruction']}")

---

## 7 — Multi-turn session with windowed partial flush

This cell shows the windowed streaming behaviour: after every `N` user turns the
middleware fires a partial `POST /ingest/session` in a background thread, sending
only the new turns since the last flush.  The session continues normally; the app
never blocks.

In [ ]:
import litellm
from smritikosh.sdk import LiteLLMMiddleware

# Using Ollama for this demo — swap model/api_base for any other provider
MODEL    = "ollama_chat/llama3.2"
API_BASE = "http://localhost:11434"

turns = [
    "I started a new project this week — a CLI tool for comparing LLM outputs.",
    "It's written in Python. I'm using Rich for the terminal UI.",
    "I want to support OpenAI, Anthropic, and Gemini out of the box.",  # flush triggers here (turn 3)
    "The hardest part is normalising the response schemas across providers.",
    "That's exactly why I'm wrapping LiteLLM underneath — it handles all that.",
    "I might open-source it if the internal team finds it useful.",  # flush triggers here (turn 6)
]

with LiteLLMMiddleware(
    litellm,
    smritikosh_url=SMRITIKOSH_URL,
    smritikosh_api_key=SMRITIKOSH_API_KEY,
    user_id=USER_ID,
    app_id=APP_ID,
    extract_every_n_turns=3,  # partial flush every 3 user turns
    use_trigger_filter=True,  # skip extraction when no high-signal phrases detected (saves LLM cost)
    session_id="notebook-multiturn-demo",  # fixed ID → idempotent re-runs
) as mw:
    history = []
    for turn in turns:
        history.append({"role": "user", "content": turn})
        response = mw.completion(
            model=MODEL,
            api_base=API_BASE,
            messages=history,
        )
        assistant_reply = response.choices[0].message.content
        history.append({"role": "assistant", "content": assistant_reply})
        print(f"User: {turn}")
        print(f"Assistant: {assistant_reply[:150]}...\n")
    # __exit__ flushes the final window (turns 7+ if any, otherwise a no-op)

print("Session ended — all turns flushed.")

---

## Provider reference

| Provider | Model string | Extra kwargs |
|---|---|---|
| Ollama (local) | `ollama_chat/<model>` | `api_base="http://localhost:11434"` |
| vLLM | `openai/<model>` | `api_base="http://localhost:8000/v1"`, `api_key="not-needed"` |
| llama.cpp | `openai/<alias>` | `api_base="http://localhost:8080/v1"`, `api_key="not-needed"` |
| Google Gemini | `gemini/gemini-1.5-flash` | `api_key=GEMINI_API_KEY` |
| OpenAI | `gpt-4o-mini` | `api_key=OPENAI_API_KEY` |
| Anthropic Claude | `claude-haiku-4-5-20251001` | `api_key=ANTHROPIC_API_KEY` |
| Azure OpenAI | `azure/<deployment>` | `api_base=...`, `api_key=...`, `api_version=...` |

## Key parameters

| Parameter | Default | Description |
|---|---|---|
| `extract_every_n_turns` | `10` | Fire a partial ingest after this many user turns. Set `0` to only flush on `close()`. |
| `use_trigger_filter` | `True` | Skip LLM extraction when no high-signal phrases are detected. Reduces cost on low-signal windows. |
| `auto_inject` | `False` | Fetch and prepend memory context before each LLM call. |
| `enable_remember_tool` | `True` | Inject the `remember()` tool into every call; intercept and save fact calls transparently. |
| `session_id` | `uuid4()` | Idempotency key — pass the same value to safely re-run a session without duplicating turns. |
| `app_id` | `"default"` | Multi-tenant namespace — memories are scoped per `(user_id, app_id)`. |

## Where to look next

- `sample/middleware_demo.py` — same patterns using a **fake OpenAI client** (no API key required)
- `sample/passive_extraction_demo.py` — direct `POST /ingest/session` calls
- `smritikosh/sdk/middleware.py` — full source of `SmritikoshMiddleware` and `LiteLLMMiddleware`
- Dashboard → Review page — approve or dismiss auto-extracted memories
- Dashboard → Identity page — see the Neo4j fact graph built from these conversations
